In [23]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *
from locallib.box import *

from lib.tables.IngesterTables import *
from lib.tables.KPI_Tables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

#Ingester Processes
from lib.ingester.ReportSummaryIngester import ReportSummaryIngester
from lib.ingester.SurveySummaryIngester import SurveySummaryIngester
from lib.ingester.EmissionSourceSummaryIngester import EmissionSourceSummaryIngester
from lib.ingester.PeakSATIngester import PeakSATIngester

from lib.kpi_processor.KPIReport import KPIReport, KPIEmissionSource, KPISurveySummary, KPIPeakSAT, KPIPOR

from datetime import date
from datetime import timedelta

In [25]:
customer_name =' Westnetz'

In [26]:
report_summary_df = Query(query="SELECT * FROM KPI_ReportSummary WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = 'Westnetz')").execute(KPIHub_Conn)
display(report_summary_df)

,ReportId,CustomerId,ReportName,ReportDate,ReportYear,ReportMonth,ReportWeek,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,...,ServicePipeKm,ServicePipeCoveredKm,BoundaryName,BoundaryType,BoundaryMode,BoundaryPlant,BoundarySubplant,BoundaryRegion,BoundarySubRegion,LastUpdated
0,FC5086EC-A25D-56E2-6211-3A20EF847249,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,CR-FC5086,2026-04-30 07:43:19.113000,2026,4,18,26.655462,26.420200,18.221517,...,8.433945,8.215250,B26R1 - RZ Westliches Rheinland - Bedburg #01,Boundaries 2026 R1,customer boundary,RZ_Westliches_Rheinland,Bedburg,None,None,2026-07-13 05:57:55.760240
1,AA0B65B3-5DCD-28EE-E5C5-3A20EF8577AA,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,CR-AA0B65,2026-04-30 07:44:26.027000,2026,4,18,53.955638,52.021254,34.769079,...,19.186559,18.283261,B26R1 - RZ Westliches Rheinland - Bedburg #02,Boundaries 2026 R1,customer boundary,RZ_Westliches_Rheinland,Bedburg,None,None,2026-07-13 05:57:55.760240
2,672697F0-3652-A300-3D21-3A2103EAF2BE,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,CR-672697,2026-05-04 06:47:40.990000,2026,5,19,51.554193,49.612014,30.757502,...,20.796691,20.238462,B26R1 - RZ Oestliches Ruhrgebiet - Selm #01,Boundaries 2026 R1,customer boundary,RZOestliches_Ruhrgebiet,Selm,None,None,2026-07-13 05:57:55.760240
3,77C2F227-6AE4-2CC8-35E1-3A2103F70BA5,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,CR-77C2F2,2026-05-04 07:00:53.797000,2026,5,19,16.357955,15.766509,11.572134,...,4.785821,4.750237,B26R1 - RZ Westliches Rheinland - Bedburg #03,Boundaries 2026 R1,customer boundary,RZ_Westliches_Rheinland,Bedburg,None,None,2026-07-13 05:57:55.760240
4,9911EDD4-10E4-88DC-CCA4-3A21091BF3C8,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,CR-9911ED,2026-05-05 06:59:18.600000,2026,5,19,35.841708,35.782984,23.391438,...,12.450270,12.412397,B26R1 - RZ Westliches Rheinland - Bedburg #04,Boundaries 2026 R1,customer boundary,RZ_Westliches_Rheinland,Bedburg,None,None,2026-07-13 05:57:55.760240
5,F6337F10-F73A-1E02-81EC-3A210F2FD214,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,CR-F6337F,2026-05-06 11:18:43.990000,2026,5,19,49.974823,49.562558,30.485562,...,19.489261,19.359410,B26R1 - RZ Oestliches Ruhrgebiet - Selm #03,Boundaries 2026 R1,customer boundary,RZOestliches_Ruhrgebiet,Selm,None,None,2026-07-13 05:57:55.760240
6,D098F1FD-FF2B-BB38-4DEF-3A21139E363C,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,CR-D098F1,2026-05-07 07:57:47.453000,2026,5,19,38.750295,38.248429,27.070543,...,11.679752,11.380880,B26R1 - RZ Westliches Rheinland - Bedburg #05,Boundaries 2026 R1,customer boundary,RZ_Westliches_Rheinland,Bedburg,None,None,2026-07-13 05:57:55.760240
7,C13E0BEB-86D3-51B8-4157-3A21139F70A0,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,CR-C13E0B,2026-05-07 07:59:07.937000,2026,5,19,11.397988,11.225958,8.603449,...,2.794539,2.737348,B26R1 - RZ Westliches Rheinland - Elsdorf #01,Boundaries 2026 R1,customer boundary,RZ_Westliches_Rheinland,Elsdorf,None,None,2026-07-13 05:57:55.760240
8,0478EFDA-9BD8-2114-3A17-3A2113A0ACBF,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,CR-0478EF,2026-05-07 08:00:28.863000,2026,5,19,48.968193,47.853076,32.862702,...,16.105491,15.668681,B26R1 - RZ Oestliches Ruhrgebiet - Selm #02,Boundaries 2026 R1,customer boundary,RZOestliches_Ruhrgebiet,Selm,None,None,2026-07-13 05:57:55.760240
9,D52355E0-C6EC-7E8C-23BF-3A2127DBDB73,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,CR-D52355,2026-05-11 06:17:31.763000,2026,5,20,14.232643,13.380510,9.615524,...,4.617118,4.272581,B26R1 - RZ Oestliches Ruhrgebiet - Selm #04,Boundaries 2026 R1,customer boundary,RZOestliches_Ruhrgebiet,Selm,None,None,2026-07-13 05:57:55.760240


In [27]:
KPI_Customer.query_table({'db_path': DB_PATH})

,CustomerId,Name,ShortName,DBLocation,LastUpdated
0,BD4D080B-1D12-D329-ABD0-39FEB9804E98,Cadent,Cadent,EU2,2026-07-13 04:06:23.072760
1,B15EED6C-3E86-302A-357F-3A182BA15005,Gas Networks Ireland,GasNetworksIreland,EU1,2026-07-13 04:06:22.967252
2,ED346655-AAF7-B2E7-43E2-3A10E455E700,Avacon,Avacon,EU1,2026-07-13 04:06:22.580255
3,C6565AAF-5251-1DBE-8D39-3A1F45B580A9,Westnetz,Westnetz,EU1,2026-07-13 04:06:22.619583
4,93A15FC7-DCB8-4BE9-1224-3A1F698B9C1F,Syna,Syna,EU1,2026-07-13 04:06:22.657353
5,A1E8BEC0-6A89-9454-137B-3A0A41825A09,NBB,NBB,EU1,2026-07-13 04:06:22.696725
6,7D9E66E1-6EE8-27D6-76B4-3A10E451DD30,EWE,EWE,EU1,2026-07-13 04:06:22.729093
7,8AABFD1C-862E-CF1E-5545-3A0EF16CBB61,ENBW,ENBW,EU1,2026-07-13 04:06:22.760779
8,B39ACD5B-3AB6-42D2-78D5-3A10F69CAA68,Thuega Energienetze,ThuegaEnergienetze,EU1,2026-07-13 04:06:22.793192
9,5DDE747E-0AAD-B45F-8039-3A1897DBC43F,Energieversorgung Filstal,EnergieversorgungFilstal,EU1,2026-07-13 04:06:22.825137


In [28]:
customer_list = get_customer_list(KPIHub_Conn)
customer_row = customer_list[customer_list['Name'] == 'Westnetz']
if not customer_row.empty:
    customer = customer_row.iloc[0]
    arguments = {'conn': KPIHub_Conn}

    reportIngester = ReportSummaryIngester(arguments)
    reportIngester.set_customer_info(customer)
    reportIngester.update_check()
    reportIngester.query_data()
    reportIngester.push_data()
    reportIngester.sanity_check()

    surveyIngester = SurveySummaryIngester(arguments)
    surveyIngester.set_customer_info(customer)
    surveyIngester.update_check()
    surveyIngester.query_data()
    surveyIngester.push_data()
    surveyIngester.sanity_check()

    emissionSourceIngester = EmissionSourceSummaryIngester(arguments)
    emissionSourceIngester.set_customer_info(customer)
    emissionSourceIngester.update_check()
    emissionSourceIngester.query_data()
    emissionSourceIngester.push_data()
    emissionSourceIngester.sanity_check()
else:
    print("Customer 'Westnetz' not found in customer list.")


In [29]:
aggregator = {'BoundaryRegion': 'BoundaryRegion'}

#Set the regional KPI
print("Setting the regional KPI")
customer_name = customer['Name']  # Fix: get correct customer name from customer row
reportKPI = KPIReport(customer_name)
emissionSourceKPI = KPIEmissionSource(customer_name)
surveyKPI = KPISurveySummary(customer_name)
if customer_name == 'Cadent':
    peakSATKPI = KPIPeakSAT(customer_name)
    KPI_List = [reportKPI, emissionSourceKPI, surveyKPI, peakSATKPI]
else:
    KPI_List = [reportKPI, emissionSourceKPI, surveyKPI]
for KPI in KPI_List:
    print(KPI.name)
    KPI.query_table()
    KPI.process_data()
    KPI.push_data()

Setting the regional KPI
KPIReport
KPIEmissionSource
KPISurveySummary


In [32]:
KPI.data['output']

,Year,PeriodValue,KPIId,Value,Id,PeriodType,LastUpdated,CustomerId
0,2026,18,SurveyDurationHours,23.79,SurveyDurationHours_Westnetz_Y2026_W18,Week,2026-07-13 06:18:50.505130,C6565AAF-5251-1DBE-8D39-3A1F45B580A9
1,2026,19,SurveyDurationHours,165.01,SurveyDurationHours_Westnetz_Y2026_W19,Week,2026-07-13 06:18:50.505130,C6565AAF-5251-1DBE-8D39-3A1F45B580A9
2,2026,20,SurveyDurationHours,97.61,SurveyDurationHours_Westnetz_Y2026_W20,Week,2026-07-13 06:18:50.505130,C6565AAF-5251-1DBE-8D39-3A1F45B580A9
3,2026,21,SurveyDurationHours,47.19,SurveyDurationHours_Westnetz_Y2026_W21,Week,2026-07-13 06:18:50.505130,C6565AAF-5251-1DBE-8D39-3A1F45B580A9
4,2026,22,SurveyDurationHours,20.09,SurveyDurationHours_Westnetz_Y2026_W22,Week,2026-07-13 06:18:50.505130,C6565AAF-5251-1DBE-8D39-3A1F45B580A9
...,...,...,...,...,...,...,...,...
148,2026,22,DayRatio,1.61,DayRatio_Westnetz_Y2026_W22,Week,2026-07-13 06:18:50.505130,C6565AAF-5251-1DBE-8D39-3A1F45B580A9
149,2026,24,DayRatio,2.68,DayRatio_Westnetz_Y2026_W24,Week,2026-07-13 06:18:50.505130,C6565AAF-5251-1DBE-8D39-3A1F45B580A9
150,2026,25,DayRatio,1.99,DayRatio_Westnetz_Y2026_W25,Week,2026-07-13 06:18:50.505130,C6565AAF-5251-1DBE-8D39-3A1F45B580A9
151,2026,27,DayRatio,0.64,DayRatio_Westnetz_Y2026_W27,Week,2026-07-13 06:18:50.505130,C6565AAF-5251-1DBE-8D39-3A1F45B580A9


In [35]:
KPI_Data.query_table({'db_path': DB_PATH})

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_Cadent_Y2023_W14,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Week,14,None,None,2026-07-13 06:03:18.741181
1,FOVMain_Cadent_Y2023_W15,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Week,15,None,None,2026-07-13 06:03:18.741181
2,FOVMain_Cadent_Y2023_W16,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Week,16,None,None,2026-07-13 06:03:18.741181
3,FOVMain_Cadent_Y2023_W17,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Week,17,None,None,2026-07-13 06:03:18.741181
4,FOVMain_Cadent_Y2023_W19,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Week,19,None,None,2026-07-13 06:03:18.741181
...,...,...,...,...,...,...,...,...,...,...
106765,DayRatio_Wales and West Utilities_Y2026_BSwans...,DayRatio,027114F8-DDB7-C0D0-1398-3A173A08C9BE,Swansea,2026,Week,24,11.19,None,2026-07-13 06:03:48.630732
106766,DayRatio_Wales and West Utilities_Y2026_BSwans...,DayRatio,027114F8-DDB7-C0D0-1398-3A173A08C9BE,Swansea,2026,Week,25,23.9,None,2026-07-13 06:03:48.630732
106767,DayRatio_Wales and West Utilities_Y2026_BSwans...,DayRatio,027114F8-DDB7-C0D0-1398-3A173A08C9BE,Swansea,2026,Week,26,32.62,None,2026-07-13 06:03:48.630732
106768,DayRatio_Wales and West Utilities_Y2026_BTorba...,DayRatio,027114F8-DDB7-C0D0-1398-3A173A08C9BE,Torbay,2026,Week,27,39.62,None,2026-07-13 06:03:48.630732
